# Create MITRE ATT&CK Tactics & Techniques json files

Download MITRE ATT&CK taxonomy data (most recent **enterprise-attack**, **ics-attack** & **mobile-attack** json files) from: https://github.com/mitre-attack/attack-stix-data.

Make sure the downloaded taxonomy data files are available in the mitre_attack_files folder.

In [1]:
from mitreattack.stix20 import MitreAttackData
from collections import defaultdict
import json

In [2]:
mitre_enterprise_file_path = "mitre_attack_files/enterprise-attack-19.1.json"
mitre_ics_file_path = "mitre_attack_files/ics-attack-19.1.json"
mitre_mobile_file_path = "mitre_attack_files/mobile-attack-19.1.json"

In [3]:
# Pre-process information about dettect-specific MITRE data components (which are called data sources in DeTT&CT).
# The file is sourced from: https://github.com/rabobank-cdc/DeTTECT/blob/master/data/dettect_data_sources.json
with open('dettect_files/dettect_data_sources.json', ) as json_data:
    dettect_data = json.load(json_data)

# Get all unique data components.
dettect_data_components = set()
for x in dettect_data:
    dettect_data_components.update(x['dettect_data_sources'])

# Create a dict mapping technique ids to names on Enterprise domain.
enterprise = MitreAttackData(mitre_enterprise_file_path)
techniques = enterprise.get_techniques(remove_revoked_deprecated=True)
techniques_id_name_dict = {x['external_references'][0]['external_id']: x['name'] for x in techniques}

# Create a dict for all dettect data components and their techniques.
dettect_data_components_dict = {}
for component in dettect_data_components:
    dettect_data_components_dict[component] = []

# Fill the dettect data components dict.
for x in dettect_data:
    for data_component in x['dettect_data_sources']:
        dettect_data_components_dict[data_component].append({'id': x['technique_id'], 'name': techniques_id_name_dict[x['technique_id']]})

# Create and fill a dict with all dettect techniques and their data components.
dettect_all_techniques_dict = {}
for x in dettect_data:
    dettect_all_techniques_dict[x['technique_id']] = x['dettect_data_sources']

# Create dettect dictionaries with just the parent techniques and just the subtechniques.
# dettect_techniques_dict = {k:v for (k,v) in dettect_all_techniques_dict.items() if not '.' in k}
# dettect_subtechniques_dict = {k:v for (k,v) in dettect_all_techniques_dict.items() if '.' in k}

In [4]:
def get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique_stix_id):

    # get data components detecting technique
    datacomponents_detects_technique = mitre_attack_data.get_datacomponents_detecting_technique(technique_stix_id)

    # Get all data sources and data components.
    data_sources = []
    data_components = []
    for d in datacomponents_detects_technique:
        datacomponent = d["object"]
        datasource = mitre_attack_data.get_object_by_stix_id(datacomponent.x_mitre_data_source_ref)
        data_sources.append(datasource.name)
        data_components.append(datacomponent.name)

    # Remove duplicates in the data sources.
    data_sources = list(set(data_sources))
    
    # Remove duplicates in the data components.
    data_components = list(dict.fromkeys(data_components))
    

    return data_sources, data_components

In [5]:
def process_mitre_attack_json(json_file_path):
    """Process the ATT&CK data from a single ATT&CK json file."""
    
    # Initialize the MitreAttackData class
    mitre_attack_data = MitreAttackData(json_file_path)
    
    # Fetch all tactics
    tactics = mitre_attack_data.get_tactics_by_matrix()
    
    # Prepare the structure for the JSON
    tactics_entries = []

    # Create a list of all platforms (will have many duplicates)
    all_platforms = []
    
    # We assume that our ATT&CK JSON files contain a single domain, which we extract here
    if 'Enterprise ATT&CK' in list(tactics.keys()):
        domain_str = 'Enterprise ATT&CK'
        domain = 'enterprise-attack'
    # We ignore the 'Network-Based Effects' key that appears in the tactics.keys() for the Mobile domain
    if 'Mobile ATT&CK' in list(tactics.keys()):
        domain_str = 'Mobile ATT&CK'
        domain = 'mobile-attack'
    if 'ATT&CK for ICS' in list(tactics.keys()):
        domain_str = 'ATT&CK for ICS'
        domain = 'ics-attack'
    


    #### Iterate over all tactics
    for tactic in tactics[domain_str]:
        # Fetch all techniques for the current tactic
        techniques = mitre_attack_data.get_techniques_by_tactic(tactic['x_mitre_shortname'], domain, remove_revoked_deprecated=True)

        # Prepare the techniques list with sub-techniques nested under their parent techniques
        techniques_dict = {}
        for technique in techniques:
            
            # # Process data sources strings into separate 'data sources' and 'data components' labels.
            # data_sources_string = technique.get('x_mitre_data_sources')
            # data_sources = []
            # data_components = []
            # if data_sources_string != None:
            #     for x in data_sources_string:
            #         data_source, data_component = x.split(': ')
            #         data_sources.append(data_source)
            #         data_components.append(data_component)
            #     data_sources = list(set(data_sources))
            #     data_components = list(set(data_components))

            platforms = technique.get('x_mitre_platforms')
            if platforms != None:
                all_platforms.extend(platforms)
            data_sources, data_components = get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique.id)
            groups = mitre_attack_data.get_groups_using_technique(technique.id)
            groups = [x['object']['name'] for x in groups]
            occurrence_groups = len(groups)

            software = mitre_attack_data.get_software_using_technique(technique.id)
            software = [x['object']['name'] for x in software]
            occurrence_software = len(software)
            
            technique_entry = {
                "name": technique['name'],
                "external_id": technique['external_references'][0]['external_id'],
                "platforms": platforms,
                'groups': groups,
                'occurrence_groups': occurrence_groups,
                'software': software,
                'occurrence_software': occurrence_software,
                'occurrence_total': occurrence_groups + occurrence_software,
                'data_sources': data_sources,
                'data_components': data_components,
                "visibility": False,  # Set default visibility to False, can be modified as needed.
                "visibility_ratio": 0,
            }

            # Add dettect-specific data sources information to all techniques and sub-techniques, for Enterprise domain.
            if domain == 'enterprise-attack':
                if technique_entry['external_id'] in dettect_all_techniques_dict:
                    technique_entry['data_components'].extend(dettect_all_techniques_dict[technique_entry['external_id']])
                    # Remove duplicates while preserving order
                    technique_entry['data_components'] = list(dict.fromkeys(technique_entry['data_components']))

            # Nest sub-techniques under their parent techniques correctly, and add parent techniques to the main techniques_dict.
            if technique.get('x_mitre_is_subtechnique'):
                parent_id = technique['external_references'][0]['external_id'].split('.')[0]
                if parent_id in techniques_dict:
                    if 'sub_techniques' not in techniques_dict[parent_id]:
                        techniques_dict[parent_id]['sub_techniques'] = []
                    techniques_dict[parent_id]['sub_techniques'].append(technique_entry)
                else:
                    techniques_dict[parent_id] = {
                        "sub_techniques": [technique_entry]
                    }
            else:
                techniques_dict[technique_entry['external_id']] = technique_entry
    
        # Sort the techniques alphabetically by name
        techniques_list = sorted([value for value in techniques_dict.values() if 'name' in value], key=lambda x: x['name'])
    
        # Sort the sub-techniques alphabetically by name
        for technique in techniques_list:
            if 'sub_techniques' in technique:
                technique['sub_techniques'] = sorted(technique['sub_techniques'], key=lambda x: x['name'])

        # Compute (sub)technique counts for the tactic.
        technique_count = len(techniques_list)
        subtechnique_count = sum([len(x['sub_techniques']) for x in techniques_list if 'sub_techniques' in x])
        all_technique_count = technique_count + subtechnique_count

        # Prepare the tactic entry
        tactic_entry = {
            "name": tactic['name'],
            "external_id": tactic['external_references'][0]['external_id'],
            "techniques": techniques_list,
            "technique_count": technique_count,
            "subtechnique_count": subtechnique_count,
            "all_technique_count": all_technique_count,
        }
        
        tactics_entries.append(tactic_entry)

    
    #### Pre-process the tactics_entries, so we can compute an order for the occurrence.
    total_techniques_count = 0
    for tactic_entry in tactics_entries:
        total_techniques_count += len(tactic_entry['techniques'])

    # Put all techniques in a single list.
    all_techniques = []
    for tactic_entry in tactics_entries:
        all_techniques += tactic_entry['techniques']

    # Assign an occurrence value based on a techniques occurrence frequency.
    # We gather all the frequencies, and perform normalization based on the position of the frequency
    # value in the total sorted list of frequency values. The normalized values are assigned to techniques.
    def assign_normalized_frequencies(occurrence_key):
        frequency_dict = defaultdict(list)
        for technique in all_techniques:
            freq_key = technique[occurrence_key]
            frequency_dict[freq_key].append(technique)

        sorted_frequency_dict = dict(sorted(frequency_dict.items()))

        # Assign the normalized frequency value to each technique.
        for freq, techniques in sorted_frequency_dict.items():
            normalized_freq_value = freq / len(sorted_frequency_dict)
            for technique in techniques:
                technique[f'{occurrence_key}_order_normalized'] = normalized_freq_value

    # Apply the function for each occurrence key.
    assign_normalized_frequencies('occurrence_total')
    assign_normalized_frequencies('occurrence_groups')
    assign_normalized_frequencies('occurrence_software')
    
    
    ### Create platforms entries
    platforms_entries = []
    
    # Remove duplicates in all_platforms list
    all_platforms = list(set(all_platforms))

    # Loop over all platforms
    for platform in all_platforms:
        techniques_by_platform = mitre_attack_data.get_techniques_by_platform(platform, remove_revoked_deprecated=True)
        all_techniques = [x.name for x in techniques_by_platform]
        techniques = [x.name  for x in techniques_by_platform if x.x_mitre_is_subtechnique == False]
        subtechniques = [x.name  for x in techniques_by_platform if x.x_mitre_is_subtechnique == True]

        # techniques_used_by_group[0]['object'].x_mitre_is_subtechnique

        # Prepare the platform entry
        platform_entry = {
            "name": platform,
            "active_in_filter": True,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        platforms_entries.append(platform_entry)

    # Sort platform entries by technique_count
    platforms_entries = sorted(platforms_entries, key=lambda item: item['all_technique_count'], reverse=True)


    
    #### Create groups entries
    groups_entries = []

    # Get all groups
    groups = mitre_attack_data.get_groups(remove_revoked_deprecated=True)

    # Loop over all groups
    for group in groups:
        techniques_used_by_group = mitre_attack_data.get_techniques_used_by_group(group.id)
        all_techniques = [x['object'].name for x in techniques_used_by_group]
        techniques = [x['object'].name for x in techniques_used_by_group if x['object'].x_mitre_is_subtechnique == False]
        subtechniques = [x['object'].name for x in techniques_used_by_group if x['object'].x_mitre_is_subtechnique == True]

        # Prepare group entry
        group_entry = {
            "name": group.name,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        groups_entries.append(group_entry)

    # Sort group entries by technique_count.
    groups_entries = sorted(groups_entries, key=lambda item: item['all_technique_count'], reverse=True)


    
    #### Create software entries
    software_entries = []

    # Get all softwares
    softwares = mitre_attack_data.get_software(remove_revoked_deprecated=True)

    # Loop over all softwares
    for software in softwares:
        techniques_used_by_software = mitre_attack_data.get_techniques_used_by_software(software.id)
        all_techniques = [x['object'].name for x in techniques_used_by_software]
        techniques = [x['object'].name for x in techniques_used_by_software if x['object'].x_mitre_is_subtechnique == False]
        subtechniques = [x['object'].name for x in techniques_used_by_software if x['object'].x_mitre_is_subtechnique == True]

        # Prepare software entry
        software_entry = {
            "name": software.name,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        software_entries.append(software_entry)

    # Sort software entries by technique_count
    software_entries = sorted(software_entries, key=lambda item: item['all_technique_count'], reverse=True)
    

    
    #### Create datasources entries
    datasources_entries = []

    # Get all data sources
    datasources = mitre_attack_data.get_datasources(remove_revoked_deprecated=True)

    # Loop over all datasources
    for datasource in datasources:

        # Prepare datasource entry
        datasource_entry = {
            'name': datasource['name'],
            'active_in_filter': True,
        }
    
        datasources_entries.append(datasource_entry)


    
    #### Create datacomponent entries
    datacomponents_entries = []

    # Get all data components
    datacomponents = mitre_attack_data.get_datacomponents(remove_revoked_deprecated=True)

    # Loop over all datacomponens
    for datacomponent in datacomponents:

        # Get all techniques detected by each datacomponent
        techniques_detected_by_datacomponent = mitre_attack_data.get_techniques_detected_by_datacomponent(datacomponent['id'])
        all_techniques = [x['object'].name for x in techniques_detected_by_datacomponent]
        techniques = [x['object'].name for x in techniques_detected_by_datacomponent if x['object'].x_mitre_is_subtechnique == False]
        subtechniques = [x['object'].name for x in techniques_detected_by_datacomponent if x['object'].x_mitre_is_subtechnique == True]

        # Prepare datacomponent entry
        datacomponent_entry = {
            "name": datacomponent['name'],
            "active_in_filter": True,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
            # 'detected_techniques': [x['object']['name'] for x in techniques_detected_by_datacomponent],
            "visibility": False,
            "quality": {
                "device_completeness": None,
                "data_field_completeness": None,
                "timeliness": None,
                "consistency": None,
                "retention": None,
            }
        }
    
        datacomponents_entries.append(datacomponent_entry)


    # Add dettect specific data components, for Enterprise domain.
    if domain == 'enterprise-attack':
      # Track which data component names we've already added from MITRE ATT&CK to avoid duplicates
      existing_component_names = {entry['name'] for entry in datacomponents_entries}
    
      for key, val in dettect_data_components_dict.items():
          # Skip if this data component already exists (from MITRE ATT&CK)
          if key in existing_component_names:
              continue
    
          all_techniques = [x['name'] for x in val]
          techniques = [x['name'] for x in val if not '.' in x['id']]
          subtechniques = [x['name'] for x in val if '.' in x['id']]
    
          # Prepare datacomponent entry
          datacomponent_entry = {
              "name": key,
              "active_in_filter": True,
              "techniques": techniques,
              "subtechniques": subtechniques,
              "all_techniques": all_techniques,
              "technique_count": len(techniques),
              "subtechnique_count": len(subtechniques),
              "all_technique_count": len(all_techniques),
              "visibility": True,
              "quality": {
                  "device_completeness": None,
                  "data_field_completeness": None,
                  "timeliness": None,
                  "consistency": None,
                  "retention": None,
              }
          }
    
          datacomponents_entries.append(datacomponent_entry)
    
    
    # Sort data components entries by technique_count.
    datacomponents_entries = sorted(datacomponents_entries, key=lambda item: item['all_technique_count'], reverse=True)

    
    #### Combine all data for the domain into a single entry.
    domain_entry = {
        "tactics": tactics_entries,
        "platforms": platforms_entries,
        "data_sources": datasources_entries,
        "data_components": datacomponents_entries,
        "groups": groups_entries,
        "softwares": software_entries,
    }

    return domain_entry

In [6]:
def flatten(lst):
    """
    Flattens a nested list, ignoring NoneType objects.

    Parameters:
    lst (list): A list that may contain nested lists and NoneType objects.

    Returns:
    list: A flattened list with NoneType objects removed.
    """
    result = []

    def _flatten(sublist):
        for item in sublist:
            if item is None:
                continue
            if isinstance(item, list):
                _flatten(item)
            else:
                result.append(item)

    _flatten(lst)
    return result

In [7]:
def convert_list(attribute_list):
    return [{"name": name, "active_in_filter": active_in_filter} for name, active_in_filter in list(zip(attribute_list, [True]*len(attribute_list)))]

In [8]:
# Load JSON files of all ATT&CK domains.
enterprise = process_mitre_attack_json(mitre_enterprise_file_path) # IMPORTANT!! When processing new ATTACK files, we may also have to update the 3rd cell in this script, handling the dettect specific data sources (in case the file dettect_data_sources.json has changed on https://github.com/rabobank-cdc/DeTTECT/tree/master/data).
ics = process_mitre_attack_json(mitre_ics_file_path)
mobile = process_mitre_attack_json(mitre_mobile_file_path)

# Prepare the final JSON structure.
final_json = {
    "enterprise": enterprise,
    "ics": ics,
    "mobile": mobile,
}
    
# Write the JSON to a file.
with open("../frontend/public/tactics_and_techniques_by_domain.json", "w") as json_file:
    json.dump(final_json, json_file, indent=2)

print("JSON file created successfully!")

JSON file created successfully!


# TESTS BELOW

In [9]:
# The functions get_techniques_detected_by_datacomponent and get_datacomponents_detecting_technique seem to return
# empty arrays when processing Mitre Attack 18.0 and 18.1 json files using the MitreAttackData package.


# Initialize the MitreAttackData class
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-17.1.json")

# Fetch all tactics
tactics = mitre_attack_data.get_tactics_by_matrix()

# Prepare the structure for the JSON
tactics_entries = []

# Create a list of all platforms (will have many duplicates)
all_platforms = []

# We assume that our ATT&CK JSON files contain a single domain, which we extract here
if 'Enterprise ATT&CK' in list(tactics.keys()):
    domain_str = 'Enterprise ATT&CK'
    domain = 'enterprise-attack'
# We ignore the 'Network-Based Effects' key that appears in the tactics.keys() for the Mobile domain
if 'Mobile ATT&CK' in list(tactics.keys()):
    domain_str = 'Mobile ATT&CK'
    domain = 'mobile-attack'
if 'ATT&CK for ICS' in list(tactics.keys()):
    domain_str = 'ATT&CK for ICS'
    domain = 'ics-attack'

In [10]:
mitre_attack_data.get_datacomponents(remove_revoked_deprecated=True)[0]

DataComponent(type='x-mitre-data-component', id='x-mitre-data-component--a7f22107-02e5-4982-9067-6625d4a1765a', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2021-10-20T15:05:19.274Z', modified='2025-04-18T15:11:20.168Z', revoked=False, object_marking_refs=['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'], name='Network Traffic Flow', description='Summarized network packet data that captures session-level details such as source/destination IPs, ports, protocol types, timestamps, and data volume, without storing full packet payloads. This is commonly used for traffic analysis, anomaly detection, and network performance monitoring.\n\n*Data Collection Measures:*\n\n- Network Flow Logs (Metadata Collection)\n    - NetFlow \n        - Summarized metadata for network conversations (no packet payloads).\n    - sFlow (Sampled Flow Logging)\n        - Captures sampled packets from switches and routers.\n        - Used for real-time traffic monitoring and 

In [11]:
# Get first tactic
tactics[domain_str][0]['x_mitre_shortname']

'reconnaissance'

In [12]:
# Get techniques
techniques = mitre_attack_data.get_techniques_by_tactic('reconnaissance', domain, remove_revoked_deprecated=True)

In [13]:
# First technique
techniques[0]

AttackPattern(type='attack-pattern', spec_version='2.1', id='attack-pattern--09312b1a-c3c6-4b45-9844-3ccc78e5d82f', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2020-10-02T16:39:33.966Z', modified='2025-04-15T22:37:32.347Z', name='Gather Victim Host Information', description="Adversaries may gather information about the victim's hosts that can be used during targeting. Information about hosts may include a variety of details, including administrative data (ex: name, assigned IP, functionality, etc.) as well as specifics regarding its configuration (ex: operating system, language, etc.).\n\nAdversaries may gather this information in various ways, such as direct collection actions via [Active Scanning](https://attack.mitre.org/techniques/T1595) or [Phishing for Information](https://attack.mitre.org/techniques/T1598). Adversaries may also compromise sites then include malicious content designed to collect host information from visitors.(Citation: ATT ScanBox) 

In [14]:
def get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique_stix_id):

    # get data components detecting technique
    datacomponents_detects_technique = mitre_attack_data.get_datacomponents_detecting_technique(technique_stix_id)

    # Get all data sources and data components.
    data_sources = []
    data_components = []
    for d in datacomponents_detects_technique:
        datacomponent = d["object"]
        datasource = mitre_attack_data.get_object_by_stix_id(datacomponent.x_mitre_data_source_ref)
        data_sources.append(datasource.name)
        data_components.append(datacomponent.name)

    # Remove duplicates in the data sources.
    data_sources = list(set(data_sources))
    
    # Remove duplicates in the data components.
    data_components = list(dict.fromkeys(data_components))

    return data_sources, data_components

In [15]:
techniques[0].name

'Gather Victim Host Information'

In [16]:
get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, techniques[0].id)

(['Internet Scan'], ['Response Content'])

In [17]:
mitre_attack_data.get_datacomponents_detecting_technique(techniques[0].id)

[{'object': DataComponent(type='x-mitre-data-component', id='x-mitre-data-component--0dcbbf4f-929c-489a-b66b-9b820d3f7f0e', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2021-10-20T15:05:19.275Z', modified='2025-04-18T15:13:36.394Z', revoked=False, object_marking_refs=['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'], name='Response Content', description='Captured network traffic that provides details about responses received during an internet scan. This data includes both protocol header values (e.g., HTTP status codes, IP headers, or DNS response codes) and response body content (e.g., HTML, JSON, or raw data). Examples:\n\n- HTTP Scan: A web server responds to a probe with an HTTP 200 status code and an HTML body indicating the default page is accessible.\n- DNS Scan: A DNS server replies to a query with a resolved IP address for a domain, along with details like Time-To-Live (TTL) and authoritative information.\n- TCP Banner Grab: A service l

In [18]:
datacomponents = mitre_attack_data.get_datacomponents(remove_revoked_deprecated=True)
len(datacomponents)

106

In [19]:
datacomponents[0]

DataComponent(type='x-mitre-data-component', id='x-mitre-data-component--a7f22107-02e5-4982-9067-6625d4a1765a', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2021-10-20T15:05:19.274Z', modified='2025-04-18T15:11:20.168Z', revoked=False, object_marking_refs=['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'], name='Network Traffic Flow', description='Summarized network packet data that captures session-level details such as source/destination IPs, ports, protocol types, timestamps, and data volume, without storing full packet payloads. This is commonly used for traffic analysis, anomaly detection, and network performance monitoring.\n\n*Data Collection Measures:*\n\n- Network Flow Logs (Metadata Collection)\n    - NetFlow \n        - Summarized metadata for network conversations (no packet payloads).\n    - sFlow (Sampled Flow Logging)\n        - Captures sampled packets from switches and routers.\n        - Used for real-time traffic monitoring and 

In [20]:
mitre_attack_data.get_techniques_detected_by_datacomponent(datacomponents[0].id)

[{'object': AttackPattern(type='attack-pattern', spec_version='2.1', id='attack-pattern--18cffc21-3260-437e-80e4-4ab8bf2ba5e9', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2020-02-20T15:35:00.025Z', modified='2025-04-15T21:48:39.804Z', name='Application Exhaustion Flood', description='Adversaries may target resource intensive features of applications to cause a denial of service (DoS), denying availability to those applications. For example, specific features in web applications may be highly resource intensive. Repeated requests to those features may be able to exhaust system resources and deny access to the application or the server itself.(Citation: Arbor AnnualDoSreport Jan 2018)', kill_chain_phases=[KillChainPhase(kill_chain_name='mitre-attack', phase_name='impact')], revoked=False, external_references=[ExternalReference(source_name='mitre-attack', url='https://attack.mitre.org/techniques/T1499/003', external_id='T1499.003'), ExternalReference(source_

In [21]:
mitre_attack_data.get_datacomponents_detecting_technique(techniques[0].id)

[{'object': DataComponent(type='x-mitre-data-component', id='x-mitre-data-component--0dcbbf4f-929c-489a-b66b-9b820d3f7f0e', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2021-10-20T15:05:19.275Z', modified='2025-04-18T15:13:36.394Z', revoked=False, object_marking_refs=['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'], name='Response Content', description='Captured network traffic that provides details about responses received during an internet scan. This data includes both protocol header values (e.g., HTTP status codes, IP headers, or DNS response codes) and response body content (e.g., HTML, JSON, or raw data). Examples:\n\n- HTTP Scan: A web server responds to a probe with an HTTP 200 status code and an HTML body indicating the default page is accessible.\n- DNS Scan: A DNS server replies to a query with a resolved IP address for a domain, along with details like Time-To-Live (TTL) and authoritative information.\n- TCP Banner Grab: A service l

In [22]:
data_sources_string = techniques[0].get('x_mitre_data_sources')
data_sources_string

['Internet Scan: Response Content']

In [23]:
techniques[0]

AttackPattern(type='attack-pattern', spec_version='2.1', id='attack-pattern--09312b1a-c3c6-4b45-9844-3ccc78e5d82f', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2020-10-02T16:39:33.966Z', modified='2025-04-15T22:37:32.347Z', name='Gather Victim Host Information', description="Adversaries may gather information about the victim's hosts that can be used during targeting. Information about hosts may include a variety of details, including administrative data (ex: name, assigned IP, functionality, etc.) as well as specifics regarding its configuration (ex: operating system, language, etc.).\n\nAdversaries may gather this information in various ways, such as direct collection actions via [Active Scanning](https://attack.mitre.org/techniques/T1595) or [Phishing for Information](https://attack.mitre.org/techniques/T1598). Adversaries may also compromise sites then include malicious content designed to collect host information from visitors.(Citation: ATT ScanBox) 